# Unity Catalog — The Full Model (Hands-On)
### Managed & External Tables · Lineage · Tagging · Row Filters & Column Masks (3 ways) · ABAC · PII Handling

Companion notebook to `Unity_Catalog_Full_Model.pptx`. Continues the QuickBite QSR
domain from the optimization course — same `orders`/`stores` shape, now viewed
through a governance lens instead of a performance one.

**Runs on Databricks Free Edition.** Confirmed in this notebook:
- Managed tables, tagging, lineage, ABAC policies, and PII functions all work with
  **zero adaptation** — ABAC's compute requirement (serverless, or DBR 16.4+) is
  satisfied by Free Edition's serverless compute directly.
- **External tables need a storage credential you control** (an AWS IAM role, Azure
  service principal, or GCS service account pointing at your own cloud storage).
  Free Edition supports external locations, but there's no bucket/credential handed
  to you by default — Section 2 shows the real syntax and explains exactly what
  you'd need to actually run it.

## Setup

In [0]:
dbutils.widgets.text("catalog", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema", "uc_governance_demo", "Schema (will be created)")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

from pyspark.sql import functions as F

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

CUSTOMERS = f"{catalog}.{schema}.customers"
ORDERS = f"{catalog}.{schema}.orders"
STORES = f"{catalog}.{schema}.stores"

current_user = spark.sql("SELECT current_user() AS u").collect()[0].u
print(f"catalog.schema = {catalog}.{schema}")
print(f"current_user() = {current_user}")

## Section 0 — Unity Catalog UI Walkthrough: End-to-End Governance for a Single Table

**Goal:** Walk through every governance feature available in Catalog Explorer for the
`customers` table — inspect metadata, create tags and comments manually, apply row
filters and column masks via the UI, and verify the policies take effect.

**Why this matters:** The previous sections used SQL/Python exclusively — this
section demonstrates how analysts, data stewards, and governance teams interact with
the same Unity Catalog objects through the point-and-click interface, no code
required. Every action here has an exact SQL equivalent (shown in Sections 4–6), but
the UI is where most day-to-day classification, tagging, and policy review happens.

---

### Step 1 — Open the Table in Catalog Explorer

1. Click **Catalog** in the left sidebar
2. Navigate to **`{catalog}` → `{schema}` → `customers`**
3. The default view shows the **Overview** tab: column list, sample data preview,
   owner, location, and table properties

**What to notice:**
- The **Location** path shows this is a **managed table** (inside Unity Catalog's
  controlled storage)
- The **Owner** is whoever created the table (typically your user principal)
- **Sample Data** is a cached preview — not a live query, so it's safe to open even
  on multi-terabyte tables

---

### Step 2 — Details Tab: Schema, Properties, and Metadata

Click the **Details** tab (second tab across the top).

**Schema section:**
- Shows every column name, type, nullable flag, and any existing tags or comments
- Notice `ssn` and `email` already have the `pii` tags we applied in Section 4 via SQL

**Properties section:**
- Table-level tags (e.g., `sensitivity:high`) appear here
- Other Delta properties: `delta.minReaderVersion`, `delta.minWriterVersion`, etc.

**Metadata section:**
- Created timestamp, last modified, number of files, table format (Delta)
- Link to the underlying Delta Lake transaction log (advanced)

**Try this:**
Scroll to the `full_name` column and notice it has **no tag yet** — we'll add one
manually in Step 4.

---

### Step 3 — Lineage Tab: Automatic Dependency Graph

Click the **Lineage** tab.

**What you see:**
- A visual graph with `customers` at the center
- Downstream: `orders_enriched` (the join we built in Section 3) appears as a
  connected node
- Upstream: none (this is a base table, not derived from another table)
- The **notebook name** that wrote `customers` is shown as a blue job/notebook node

**Click on the `orders_enriched` node:**
- Catalog Explorer highlights the edge and shows the query/notebook that created the
  lineage relationship
- You can trace data flow forward (impact analysis: "what breaks if I change this
  table?") or backward (root-cause analysis: "where did this value originate?")

**No code was written to produce this graph** — Unity Catalog captured it
automatically when you ran the join cell in Section 3.

---

### Step 4 — Manually Add Tags via the UI

Stay on the **Details** tab, or return to it.

1. Find the **`full_name`** column in the schema list
2. Hover over the row → a **tag icon** (label with a +) appears on the right
3. Click **+ Add Tag**
4. In the dialog:
   - **Key:** `pii`
   - **Value:** `name`
5. Click **Add**

The tag `pii:name` now appears inline next to `full_name` in the schema table.

**What just happened behind the scenes:**
Catalog Explorer executed the equivalent of:
```sql
ALTER TABLE {catalog}.{schema}.customers
ALTER COLUMN full_name SET TAGS ('pii' = 'name');
```
You didn't write that SQL — the UI wrote it for you.

**Repeat for the table-level tag:**
1. Scroll to the **Properties** section on the Details tab
2. Click **+ Add Tag** (table-level, not column-level)
3. Key: `department`, Value: `customer_success`
4. Click **Add**

Now the table itself carries the tag `department:customer_success`, visible in the
Properties panel and queryable via `system.information_schema.table_tags`.

---

### Step 5 — Manually Add a Column Comment

Comments are the human-readable descriptions that appear in Catalog Explorer search
results and schema previews.

1. Find the **`region`** column
2. Hover → click the **pencil/edit icon** next to the column name
3. In the **Description** field, type:
   ```
   Geographic sales region (North, South, East, West) for territory assignment
   ```
4. Click **Save**

The comment now appears beneath the column name in the Details view, and the column
will show this description in autocomplete/intellisense when queried.

**SQL equivalent:**
```sql
COMMENT ON COLUMN {catalog}.{schema}.customers.region IS
  'Geographic sales region (North, South, East, West) for territory assignment';
```

---

### Step 6 — Permissions Tab: View and Grant Access

Click the **Permissions** tab.

**What you see:**
- A list of principals (users, groups, service principals) and their current grants:
  `SELECT`, `MODIFY`, `OWNER`, etc.
- Your own user principal appears with `OWNER` (you created the table)
- `account users` group may have inherited `SELECT` from the schema-level grant

**Try granting access (if you have `OWNER` or `MANAGE` on the table):**
1. Click **Grant**
2. Select a principal (e.g., `account users` or a specific user)
3. Choose privileges: **SELECT** (read), **MODIFY** (write), **SELECT and MODIFY**
4. Click **Grant**

**Revoke:**
Click the **three-dot menu** next to any grant → **Revoke**.

**This is the UI for GRANT/REVOKE statements** — the exact same privilege model
discussed in Unity Catalog's ACL documentation, just point-and-click instead of SQL.

---

### Step 7 — Sample Data Tab: Quick Data Preview (Policy-Aware)

Click the **Sample Data** tab.

**What you see:**
- The first ~1000 rows (cached preview, not a full scan)
- **If any row filters or column masks are active on this table and apply to your
  principal, the preview respects them** — you'll see masked `ssn` values and
  filtered rows exactly as a `SELECT * FROM customers` query would return

This preview is a fast way to confirm that a policy you just applied (via SQL or UI)
is working as intended, without writing a query.

---

### Step 8 — Apply a Row Filter via the UI

Unity Catalog's **Attribute-Based Access Control (ABAC)** policies (GA since late
2024) are managed via SQL (`CREATE POLICY`), but **table-level row filters** can be
applied through the UI.

**Current limitation (as of early 2025):**
The Catalog Explorer UI does **not yet expose a "Create Policy" button for ABAC**
policies — those must still be created via SQL (`CREATE POLICY ... ON SCHEMA`) as
shown in Section 5B. However, the **classic table-level `SET ROW FILTER`** syntax
(Section 5A) is supported through the UI's **Row Filter** panel on some Databricks
workspace editions.

**If your workspace shows a "Row Filter" section on the Details tab:**
1. Scroll to **Row Filter** (below the schema table)
2. Click **Add Row Filter**
3. Select or create a UDF: `{catalog}.{schema}.filter_south_classic`
   (the function we created in Section 5A)
4. Choose the column the filter operates on: `region`
5. Click **Apply**

**SQL equivalent (from Section 5A):**
```sql
ALTER TABLE {catalog}.{schema}.customers
SET ROW FILTER {catalog}.{schema}.filter_south_classic ON (region);
```

Once applied, refresh the **Sample Data** tab — you'll see only `region = 'South'`
rows if you are not in the `admins` group.

**To remove the filter via UI:**
Return to **Row Filter** section → click **Remove**.

---

### Step 9 — Apply a Column Mask via the UI

Similarly, **column masks** can be managed through the UI for the classic
table-level approach (Section 5A).

**If your workspace shows a "Column Masks" section on the Details tab:**
1. Find the **`ssn`** column in the schema list
2. Click the **three-dot menu** next to the column → **Set Mask**
3. Select the masking UDF: `{catalog}.{schema}.mask_ssn_classic`
   (created in Section 5A)
4. Click **Apply**

**SQL equivalent:**
```sql
ALTER TABLE {catalog}.{schema}.customers
ALTER COLUMN ssn SET MASK {catalog}.{schema}.mask_ssn_classic;
```

Once applied, the **Sample Data** tab will show `***-**-6789` for all `ssn` values
(assuming you are not in the `admins` group).

**To remove the mask via UI:**
Three-dot menu → **Drop Mask**.

---

### Step 10 — Policies Tab: View Active ABAC Policies (Read-Only, for now)

If your table is covered by a **schema-level ABAC policy** (like the ones we created
in Section 5B or Section 6), those policies appear on the **Policies** tab.

1. Click the **Policies** tab
2. You'll see a list of all ABAC policies that match this table's tags/columns:
   - Policy name: `mask_pii_columns`, `south_region_only`
   - Type: `COLUMN MASK` or `ROW FILTER`
   - Scope: `SCHEMA {catalog}.{schema}`
   - Condition: the tag match expression (e.g., `has_tag_value('pii', 'ssn')`)

**Current limitation:**
This tab is **read-only** in most workspace editions — you can **view** which
policies apply, but to **create or modify** an ABAC policy, you must use SQL:
```sql
CREATE POLICY <name> ON SCHEMA {catalog}.{schema}
COLUMN MASK <udf> TO <principals> FOR TABLES
MATCH COLUMNS <tag_condition> AS <alias> ON COLUMN <alias>;
```
(See Section 5B and Section 6 for full syntax.)

**Why this tab matters:**
Even if you can't create policies here, you **can discover which policies are
affecting a table** — critical for debugging "Why is this column masked?" or "Why am
I only seeing some rows?" questions.

---

### Step 11 — History Tab: Audit Trail (Table-Level Operations)

Click the **History** tab.

**What you see:**
- A chronological list of every DDL operation performed on this table:
  - `CREATE TABLE`
  - `ALTER TABLE ... SET TAGS`
  - `ALTER TABLE ... SET MASK`
  - `COMMENT ON TABLE`
  - `GRANT` / `REVOKE`
- Each entry shows:
  - Timestamp
  - Operation type
  - User principal who performed it
  - Statement text (if available)

**This is Unity Catalog's built-in audit log** — every governance action is logged
automatically, no setup required.

**Use case:** If a policy suddenly stops working, check the History tab to see if
someone dropped the mask or revoked a permission.

---

### Step 12 — Recap: What We Just Did, Entirely via UI

| Action | UI Location | SQL Equivalent |
|--------|-------------|----------------|
| View schema, sample data | Overview / Details / Sample Data tabs | `DESCRIBE TABLE`, `SELECT * LIMIT 1000` |
| Inspect lineage | Lineage tab | `system.access.table_lineage` view |
| Add a column tag | Details → column row → + Add Tag | `ALTER TABLE ... ALTER COLUMN ... SET TAGS` |
| Add a table tag | Details → Properties → + Add Tag | `ALTER TABLE ... SET TAGS` |
| Add a column comment | Details → column row → edit icon | `COMMENT ON COLUMN` |
| View permissions | Permissions tab | `SHOW GRANTS ON TABLE` |
| Grant/revoke access | Permissions → Grant / Revoke | `GRANT` / `REVOKE` |
| Apply row filter (classic) | Details → Row Filter → Add | `ALTER TABLE ... SET ROW FILTER` |
| Apply column mask (classic) | Details → column → Set Mask | `ALTER TABLE ... ALTER COLUMN ... SET MASK` |
| View active ABAC policies | Policies tab | `SHOW EFFECTIVE POLICIES ON SCHEMA` |
| View operation history | History tab | `system.access.audit` (table-level subset) |

**Key takeaway:** Every governance operation has **two interfaces** — SQL (for
automation, CI/CD, bulk operations) and UI (for exploratory work, ad-hoc tagging,
and policy inspection). They operate on the **same underlying Unity Catalog
metadata** — a tag added via UI is immediately visible to a SQL query against
`information_schema.column_tags`, and a policy created via SQL appears in the
Policies tab instantly.

---

### Step 13 — Verify the Policies Work End-to-End

Now that we've applied tags, comments, a row filter, and a column mask (either via
SQL in earlier sections or via UI in this section), **run a query as a non-admin
user** to confirm everything is enforced:

```python
# Run this cell as a user who is NOT in the 'admins' group
display(spark.table(CUSTOMERS))
```

**Expected output:**
- Only rows where `region = 'South'` (if the row filter is active)
- `ssn` column shows `***-**-XXXX` (if the column mask is active)
- `email` may also be masked if a policy targets the `pii:email` tag

**If you ARE in the 'admins' group:**
You'll see all rows and all columns unmasked — the policies we created explicitly
exempt the `admins` group.

**To test as a different user without switching accounts:**
1. Create a second test user in your Databricks account (Account Console → User
   Management → Add User)
2. Invite them to the workspace
3. Have them open this notebook and run the query above
4. Compare their output to yours

This is the **real-world governance workflow** — data stewards classify and tag via
UI, policy authors write `CREATE POLICY` statements in SQL, and end users query
tables normally, completely unaware that masking/filtering is happening under the
hood.

---

### Step 14 — Clean Up (Optional)

If you want to remove the policies and tags we applied in this section:

**Via UI:**
- Tags: Details tab → hover over tag → X icon
- Row filter: Details → Row Filter → Remove
- Column mask: Details → column three-dot menu → Drop Mask

**Via SQL:**
```sql
-- Drop ABAC policies (if you created them in Section 5B or Section 6)
DROP POLICY IF EXISTS mask_pii_columns ON SCHEMA {catalog}.{schema};
DROP POLICY IF EXISTS south_region_only ON SCHEMA {catalog}.{schema};

-- Drop table-level row filter and column mask (if you applied them in Section 5A)
ALTER TABLE {catalog}.{schema}.customers DROP ROW FILTER;
ALTER TABLE {catalog}.{schema}.customers ALTER COLUMN ssn DROP MASK;

-- Remove tags
ALTER TABLE {catalog}.{schema}.customers ALTER COLUMN full_name UNSET TAGS ('pii');
ALTER TABLE {catalog}.{schema}.customers UNSET TAGS ('department');

-- Remove comment
COMMENT ON COLUMN {catalog}.{schema}.customers.region IS NULL;
```



---



## Section 1 — Managed Tables, and the Object Hierarchy in Practice

Every table we create below lives at `{catalog}.{schema}.{table}` — the same
three-level namespace from the deck, walked in code instead of a diagram.

In [0]:
customer_rows = [
    (1, "Alice Rao", "alice@example.com", "123-45-6789", "Bengaluru", "South"),
    (2, "Ben Fernandes", "ben@example.com", "987-65-4321", "Mumbai", "West"),
    (3, "Chitra Nair", "chitra@example.com", "555-11-2222", "Chennai", "South"),
    (4, "Deepak Shah", "deepak@example.com", "444-33-1111", "Delhi", "North"),
    (5, "Elena Kapoor", "elena@example.com", "111-22-3333", "Kolkata", "East"),
]
(
    spark.createDataFrame(customer_rows, ["customer_id", "full_name", "email", "ssn", "city", "region"])
    .write.format("delta").mode("overwrite").saveAsTable(CUSTOMERS)
)

store_rows = [(sid, f"Store {sid}", ["South", "West", "North", "East"][sid % 4]) for sid in range(1, 11)]
(
    spark.createDataFrame(store_rows, ["store_id", "store_name", "region"])
    .write.format("delta").mode("overwrite").saveAsTable(STORES)
)

order_rows = [(i, (i % 5) + 1, (i % 10) + 1, round(20 + i * 3.5, 2)) for i in range(1, 51)]
(
    spark.createDataFrame(order_rows, ["order_id", "customer_id", "store_id", "order_amount"])
    .write.format("delta").mode("overwrite").saveAsTable(ORDERS)
)

display(spark.sql(f"DESCRIBE DETAIL {CUSTOMERS}").select("name", "location", "numFiles"))
print(f"\nManaged table location is inside Unity Catalog's own storage — you don't choose")
print(f"it, and DROP TABLE would delete these files along with the catalog entry.")

## Section 2 — External Tables (Syntax + What You'd Actually Need)

This is real, runnable syntax — but it **requires a storage credential pointing at
cloud storage you control**, which Free Edition doesn't provision for you
automatically. Read this section for the mechanism; treat the final cell as
reference unless you already have an AWS/Azure/GCS account with a bucket ready.

**Step 1 — a storage credential** (abstracts a long-lived cloud credential):
```sql
CREATE STORAGE CREDENTIAL quickbite_cred
  WITH (AWS_IAM_ROLE = 'arn:aws:iam::123456789012:role/quickbite-uc-access');
```

**Step 2 — an external location** (combines the credential with a specific path):
```sql
CREATE EXTERNAL LOCATION quickbite_ext_loc
  URL 's3://quickbite-raw-data/orders/'
  WITH (CREDENTIAL quickbite_cred);

GRANT READ FILES, WRITE FILES, CREATE EXTERNAL TABLE
  ON EXTERNAL LOCATION quickbite_ext_loc TO `account_users`;
```

**Step 3 — the external table itself:**
```sql
CREATE TABLE quickbite_prod.bronze.orders_external
  (order_id INT, customer_id INT, store_id INT, order_amount DOUBLE)
USING DELTA
LOCATION 's3://quickbite-raw-data/orders/';
```

**The one-line difference that matters operationally:**
```sql
DROP TABLE quickbite_prod.bronze.orders_external;
-- Removes the catalog entry only. The Parquet/Delta files under
-- s3://quickbite-raw-data/orders/ are untouched — compare this to Section 1's
-- managed CUSTOMERS table, where DROP TABLE would delete the data too.
```

## Section 3 — Data Lineage

Build a two-hop transformation, then look at what Unity Catalog captured
automatically — no lineage-specific code was written anywhere above.

In [0]:
ORDERS_ENRICHED = f"{catalog}.{schema}.orders_enriched"

(
    spark.table(ORDERS)
    .join(spark.table(CUSTOMERS), "customer_id")
    .join(spark.table(STORES), "store_id")
    .select("order_id", "full_name", "city", "store_name", "order_amount")
    .write.format("delta").mode("overwrite").saveAsTable(ORDERS_ENRICHED)
)

print(f"{ORDERS_ENRICHED} written from {ORDERS} + {CUSTOMERS} + {STORES}.")
print(f"\nOpen Catalog Explorer -> {catalog} -> {schema} -> orders_enriched -> Lineage tab.")
print("You'll see all three source tables as upstream nodes, and this notebook as the")
print("job/notebook node that produced the join — captured purely from having run the")
print("cell above, with zero lineage-specific code.")

## Section 4 — Tagging

Tag the `ssn` column now — Section 6's ABAC policies match against this exact tag.

In [0]:
spark.sql(f"ALTER TABLE {CUSTOMERS} ALTER COLUMN ssn SET TAGS ('pii' = 'ssn')")
spark.sql(f"ALTER TABLE {CUSTOMERS} ALTER COLUMN email SET TAGS ('pii' = 'email')")
spark.sql(f"ALTER TABLE {CUSTOMERS} SET TAGS ('sensitivity' = 'high')")

display(
    spark.sql(f"SELECT * FROM {catalog}.information_schema.column_tags WHERE table_name = 'customers'")
)
print(f"\nTable-level tag check: SHOW TBLPROPERTIES doesn't show tags — use")
print(f"information_schema.table_tags / column_tags, or Catalog Explorer's Tags panel.")

## Section 5 — Row Filters & Column Masks: Three Ways

Same goal each time — hide `ssn` from non-admins, restrict rows to `region = 'South'`
for non-admins — implemented three different ways so you can compare them directly.
All three can't be active on the same column simultaneously in this demo (they'd
conflict), so each is shown, then cleanly removed before the next.

### 5A — Table-Level (Classic): `ALTER TABLE ... SET ROW FILTER` / `SET MASK`

The original, per-table mechanism — still the official syntax, still the right
choice for a one-off rule on a single table.

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_ssn_classic(ssn STRING)
RETURNS STRING
RETURN CASE WHEN is_account_group_member('admins') THEN ssn ELSE CONCAT('***-**-', RIGHT(ssn, 4)) END
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.filter_south_classic(region STRING)
RETURNS BOOLEAN
RETURN is_account_group_member('admins') OR region = 'South'
""")

spark.sql(f"ALTER TABLE {CUSTOMERS} ALTER COLUMN ssn SET MASK {catalog}.{schema}.mask_ssn_classic")
spark.sql(f"ALTER TABLE {CUSTOMERS} SET ROW FILTER {catalog}.{schema}.filter_south_classic ON (region)")

print("Applied. As a non-admin caller, this query now returns only South rows, masked:")
display(spark.table(CUSTOMERS))

# Clean up before showing the next approach, so the two don't stack
spark.sql(f"ALTER TABLE {CUSTOMERS} ALTER COLUMN ssn DROP MASK")
spark.sql(f"ALTER TABLE {CUSTOMERS} DROP ROW FILTER")

### 5B — ABAC Policy (GA): tag-driven, scales across the whole schema

The same rule, but defined **once at the schema level** — it will also cover any
table created under this schema tomorrow, as long as its columns carry the matching
tag. Requires `MANAGE` on the schema (or ownership) and `EXECUTE` on the function.

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_pii_last4(val STRING)
RETURNS STRING
RETURN CONCAT('***-**-', RIGHT(val, 4))
""")

spark.sql(f"""
CREATE POLICY mask_pii_columns
ON SCHEMA {catalog}.{schema}
COMMENT 'Mask any column tagged pii:ssn for all users except admins'
COLUMN MASK {catalog}.{schema}.mask_pii_last4
TO `account users` EXCEPT `admins`
FOR TABLES
MATCH COLUMNS has_tag_value('pii', 'ssn') AS ssn_col
ON COLUMN ssn_col
""")

display(spark.sql(f"SHOW EFFECTIVE POLICIES ON SCHEMA {catalog}.{schema}"))
print("\nThis policy now applies to `customers.ssn` (tagged pii:ssn in Section 4) AND to")
print("any future table in this schema with a column carrying the same tag — no per-table")
print("ALTER TABLE needed for those future tables.")

# Clean up before showing the next approach
spark.sql(f"DROP POLICY mask_pii_columns ON SCHEMA {catalog}.{schema}")

### 5C — Dynamic View: no special privileges beyond creating a view

The most portable option — this is exactly the pattern the official DE Professional
exam guide's sample question uses. No `SET MASK`/`SET ROW FILTER`/`CREATE POLICY`
privilege required, just ordinary view-creation rights.

In [0]:
CUSTOMERS_VIEW = f"{catalog}.{schema}.customers_protected_v"

spark.sql(f"""
CREATE OR REPLACE VIEW {CUSTOMERS_VIEW} AS
SELECT
  customer_id,
  full_name,
  CASE WHEN is_account_group_member('admins') THEN email ELSE 'REDACTED' END AS email,
  CASE WHEN is_account_group_member('admins') THEN ssn ELSE CONCAT('***-**-', RIGHT(ssn, 4)) END AS ssn,
  city,
  region
FROM {CUSTOMERS}
WHERE is_account_group_member('admins') OR region = 'South'
""")

display(spark.table(CUSTOMERS_VIEW))
print("\nSame masking + filtering logic as 5A/5B, expressed entirely in view SQL.")
print("Trade-off: every consumer must know to query the VIEW, not the base table —")
print("the base table itself carries no protection unless 5A or 5B is also applied to it.")

## Section 6 — ABAC in Full: Row Filter + Column Mask Together

Re-apply the ABAC approach from 5B, this time pairing a column mask with a row
filter policy at the same time, and confirm both are active together.

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.non_south_region(region STRING)
RETURNS BOOLEAN
RETURN region = 'South'
""")

spark.sql(f"""
CREATE POLICY mask_ssn_abac
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.mask_pii_last4
TO `account users` EXCEPT `admins`
FOR TABLES
MATCH COLUMNS has_tag_value('pii', 'ssn') AS ssn_col
ON COLUMN ssn_col
""")

spark.sql(f"""
CREATE POLICY south_region_only
ON SCHEMA {catalog}.{schema}
COMMENT 'Non-admins only see South region rows on tables tagged sensitivity:high'
ROW FILTER {catalog}.{schema}.non_south_region
TO `account users` EXCEPT `admins`
FOR TABLES
WHEN has_tag_value('sensitivity', 'high')
MATCH COLUMNS has_tag('region') AS region_col
USING COLUMNS (region_col)
""")

display(spark.sql(f"SHOW EFFECTIVE POLICIES ON SCHEMA {catalog}.{schema}"))
display(spark.table(CUSTOMERS))
print("\nBoth policies are independent objects, both discoverable via SHOW EFFECTIVE")
print("POLICIES, both driven entirely by tags — nothing here references 'customers' by name.")

## Section 7 — PII Handling: Hashing, Tokenization, Anonymization, Pseudonymization

Four techniques, one column, so you can compare the actual output side by side.
**Run this against the raw table directly** (bypassing the masks/filters above) to
see true inputs and outputs — in practice you'd apply one of these transformations
*before* writing to a table others query, not as a runtime mask.

In [0]:
pii_demo = spark.createDataFrame(
    [(1, "123-45-6789", 34), (2, "987-65-4321", 28), (3, "555-11-2222", 45)],
    ["customer_id", "ssn", "age"],
)

# 1. Hashing (sha2) — one-way, deterministic
hashed = pii_demo.withColumn("ssn_sha256", F.sha2(F.col("ssn"), 256))

# 2. Tokenization — reversible via a lookup table you control (simulated here)
token_map = spark.createDataFrame([("123-45-6789", "TOK-88291-XQ"), ("987-65-4321", "TOK-14027-ZP"), ("555-11-2222", "TOK-93820-LM")], ["ssn", "token"])
tokenized = pii_demo.join(token_map, "ssn").select("customer_id", "ssn", "token")
print("Reversal requires the token_map table — store it somewhere far more locked-down")
print("than the data it protects; that separation is the entire point of tokenization.")

# 3. Anonymization — irreversible: generalization (age -> age band) + suppression (drop ssn)
anonymized = pii_demo.withColumn(
    "age_band", F.when(F.col("age") < 30, "18-29").when(F.col("age") < 45, "30-44").otherwise("45+")
).drop("ssn", "age")

# 4. Pseudonymization — replace with a stand-in, mapping kept separately (same shape as
#    tokenization technically, but the intent differs: a pseudonym is usually a synthetic
#    identity-shaped value, not an opaque token, and often permits re-identification within
#    the same dataset for join purposes without exposing the real value)
pseudo_map = spark.createDataFrame([("123-45-6789", "CUST-A1"), ("987-65-4321", "CUST-A2"), ("555-11-2222", "CUST-A3")], ["ssn", "pseudonym"])
pseudonymized = pii_demo.join(pseudo_map, "ssn").select("customer_id", "pseudonym", "age")

print("\n1. Hashing:")
display(hashed)
print("2. Tokenization:")
display(tokenized)
print("3. Anonymization:")
display(anonymized)
print("4. Pseudonymization:")
display(pseudonymized)

### 7.1 — Which one to reach for

| Need | Use |
|---|---|
| Join/group on the value without ever needing it back | **Hashing** |
| Recover the original value later, under strict access control | **Tokenization** |
| Publish data with no path back to an individual, even internally | **Anonymization** |
| Analyze patterns per-person across a dataset without exposing identity | **Pseudonymization** |

**Cert/interview framing:** the exam guide's Security & Compliance domain names
hashing, tokenization, suppression, and generalization explicitly — the table above
is close to a direct decision-map for those objective statements.

## Recap

| Concept | What you ran |
|---|---|
| Managed table | `CUSTOMERS`, `ORDERS`, `STORES` via `saveAsTable` |
| External table | Syntax shown — needs your own cloud storage credential |
| Lineage | `orders_enriched` join, inspected via Catalog Explorer |
| Tagging | `pii:ssn`, `pii:email`, `sensitivity:high` |
| Row filter / column mask — 3 ways | Table-level (`ALTER TABLE`), ABAC (`CREATE POLICY`), dynamic view |
| ABAC, both policy types together | `mask_ssn_abac` + `south_region_only` |
| PII techniques | Hashing, tokenization, anonymization, pseudonymization |

**Next — Part 2:** ACLs and the Unity Catalog permission inheritance model,
`dbutils.secrets`, and compliance-driven retention/purging pipelines.